# 33. 양자화 — 모델을 작게 만들기

> **제33장** · **이론편 대응: 22.5절 (양자화와 QLoRA)**
> **예상 소요**: 80분
> **필요 사양**: **[CPU]** 로 실행 가능
> **추가 설치**: 없음
> **다운로드**: DistilGPT-2 (28장에서 받았다면 재사용)

---

## 이 장에서 하는 일

32장에서 QLoRA를 다루며 **양자화를 개념으로만** 설명했다.
이번에는 **직접 구현하고 품질 저하를 측정한다.**

| 절 | 하는 일 | 이론편 대응 |
|---|---|---|
| 1 | 왜 양자화인가 | 22.5절 |
| 2 | **대칭 양자화 손계산 검증** ★ | 22.5절 |
| 3 | 비대칭 양자화와 그룹 단위 | 22.5절 |
| 4 | **비트 수와 품질 — 절벽이 있다** ★ | 22.5절 |
| 5 | 실제 모델 양자화 | 22.5절 |
| 6 | 어떤 층을 양자화할까 | 22.5절 |
| 7 | 실무 도구 (bitsandbytes, GGUF) | 22.5절 |
| 8 | 무엇을 얻고 무엇을 잃나 | 22.5절 |

**2절과 4절이 핵심이다.** 손계산으로 원리를 확인하고,
**8비트는 괜찮은데 6비트부터 무너지는** 현상을 직접 측정한다.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import platform
import copy
import io
import time

_c = {"Windows": ["Malgun Gothic"], "Darwin": ["AppleGothic"],
      "Linux": ["NanumGothic", "Noto Sans CJK KR", "Noto Sans CJK JP"]}
_a = {f.name for f in fm.fontManager.ttflist}
for _n in _c.get(platform.system(), []):
    if _n in _a:
        plt.rcParams["font.family"] = _n
        break
plt.rcParams["axes.unicode_minus"] = False

np.set_printoptions(precision=4, suppress=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} / 장치: {device}")

---

## 1. 왜 양자화인가 — 이론편 22.5절

**32장 9절에서 8GB GPU의 한계를 계산했다.**

$$M = N_{params} \times \frac{b}{8}\text{ bytes}$$

7B 모델을 FP16으로 올리면 13GB — 8GB GPU에 안 들어간다.
**같은 모델을 4비트로 표현하면 3.3GB**가 되어 가능해진다.

| 정밀도 | 비트 | 7B 모델 크기 | 8GB GPU |
|---|---|---|---|
| FP32 | 32 | 26 GB | 불가능 |
| FP16 | 16 | 13 GB | 불가능 |
| INT8 | 8 | 6.5 GB | 빠듯함 |
| **INT4** | **4** | **3.3 GB** | **가능** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("=" * 78)
print("정밀도별 모델 크기 (이론편 22.5절)")
print("=" * 78)

model_sizes = [0.5, 1.5, 3, 7, 13, 30, 70]
precisions = [("FP32", 4), ("FP16", 2), ("INT8", 1), ("INT4", 0.5)]

print(f"{'모델':<12}" + "".join(f"{p:<14}" for p, _ in precisions))
print("-" * 78)
for n in model_sizes:
    row = f"{n}B".ljust(12)
    for _, bytes_per in precisions:
        gb = n * bytes_per
        row += f"{gb:>7.1f} GB   "
    print(row)
print("-" * 78)
print()

USABLE_8GB = 6.8
print(f"8GB GPU 실사용 가능: 약 {USABLE_8GB} GB")
print()
print(f"{'모델':<12}{'FP16':<14}{'INT8':<14}{'INT4'}")
print("-" * 78)
for n in [1.5, 3, 7, 13]:
    row = f"{n}B".ljust(12)
    for _, bytes_per in [("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
        gb = n * bytes_per
        row += f"{'가능' if gb < USABLE_8GB else '불가능':<14}"
    print(row)
print("-" * 78)
print()
print("[핵심] 양자화는 '더 큰 모델을 쓸 수 있게' 한다")
print("  7B 를 INT4 로 쓰는 것이 1.5B 를 FP16 으로 쓰는 것보다 대체로 낫다.")
print("  품질 저하보다 모델 크기의 이득이 크기 때문이다.")

---

## 2. 대칭 양자화 ★ — 이론편 22.5절 검증

**실수를 정수로 바꾼다.** 가장 단순한 방식이 대칭 양자화다.

$$s = \frac{\max|w|}{q_{max}}, \qquad q = \text{round}\left(\frac{w}{s}\right), \qquad \hat{w} = q \times s$$

$q_{max}$는 표현 가능한 최댓값이다. 8비트 부호 있는 정수면 $2^7 - 1 = 127$.

**이론편 22.5절의 예제로 확인하자.**

In [ ]:
import numpy as np


def quantize_symmetric(w, bits=8):
    """대칭 양자화 (이론편 22.5절)

    0을 중심으로 대칭이므로 영점(zero-point)이 필요 없다.
    """
    qmax = 2 ** (bits - 1) - 1
    scale = np.abs(w).max() / qmax
    q = np.round(w / scale).clip(-qmax - 1, qmax).astype(int)
    return q, scale


def dequantize(q, scale):
    """정수를 실수로 되돌린다"""
    return q * scale


w = np.array([-0.8, -0.3, 0.0, 0.25, 0.7])

print("=" * 78)
print("대칭 양자화 — 단계별")
print("=" * 78)
print(f"원본 가중치: {w}")
print(f"최댓값(절대): {np.abs(w).max()}")
print()

for bits in [8, 4]:
    qmax = 2 ** (bits - 1) - 1
    q, scale = quantize_symmetric(w, bits)
    dq = dequantize(q, scale)
    err = np.abs(w - dq)

    print(f"[{bits}비트]")
    print(f"  표현 범위 : -{qmax+1} ~ {qmax}  ({2**bits}단계)")
    print(f"  scale     : {np.abs(w).max()} / {qmax} = {scale:.6f}")
    print()
    print(f"  {'원본':<12}{'÷ scale':<14}{'반올림':<12}{'복원':<12}{'오차'}")
    print("  " + "-" * 62)
    for wi, qi, di, ei in zip(w, q, dq, err):
        print(f"  {wi:<12.2f}{wi/scale:<14.3f}{qi:<12}{di:<12.4f}{ei:.4f}")
    print("  " + "-" * 62)
    print(f"  최대 오차: {err.max():.4f}")
    print()

# 이론편 값 검증
q8, s8 = quantize_symmetric(w, 8)
q4, s4 = quantize_symmetric(w, 4)
assert q8[0] == -127 and q8[-1] == 111
assert q4[0] == -7 and q4[-1] == 6
print("[OK] 이론편 22.5절 값과 일치")
print("  8비트: [-127, -48, 0, 40, 111]")
print("  4비트: [-7, -3, 0, 2, 6]")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

w_range = np.linspace(-1, 1, 400)

for ax, bits in zip(axes, [8, 4, 2]):
    qmax = 2 ** (bits - 1) - 1
    scale = 1.0 / qmax
    q = np.round(w_range / scale).clip(-qmax - 1, qmax)
    dq = q * scale

    ax.plot(w_range, w_range, linewidth=1.5, color="#94A3B8",
            linestyle="--", label="원본 (연속)")
    ax.plot(w_range, dq, linewidth=2, color="#1E40AF",
            label=f"{bits}비트 양자화")
    ax.set_xlabel("원본 값")
    ax.set_ylabel("복원된 값")
    ax.set_title(f"{bits}비트 ({2**bits}단계)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("비트 수와 표현 가능한 단계")
print("=" * 78)
print(f"{'비트':<10}{'단계 수':<16}{'해상도 (범위 1.0 기준)':<28}{'평균 오차(이론)'}")
print("-" * 78)
for bits in [16, 8, 6, 4, 3, 2]:
    levels = 2 ** bits
    resolution = 2.0 / levels
    print(f"{bits:<10}{levels:<16,}{resolution:<28.6f}{resolution/4:.6f}")
print("-" * 78)
print()
print("계단이 굵어질수록 원본과 멀어진다.")
print("  2비트는 4단계뿐이라 사실상 형태를 잃는다.")

---

## 3. 비대칭 양자화와 그룹 단위 — 이론편 22.5절

### 비대칭 양자화

가중치가 한쪽으로 치우쳐 있으면 대칭 방식은 낭비가 생긴다.

$$s = \frac{w_{max} - w_{min}}{2^b - 1}, \qquad z = \text{round}\left(-\frac{w_{min}}{s}\right)$$

**영점(zero-point)**을 두어 실제 범위에 맞춘다.

### 그룹 단위 양자화

가중치 전체에 하나의 scale을 쓰면, **큰 값 하나가 전체 해상도를 망친다.**
작은 그룹으로 나눠 각각 scale을 두면 훨씬 정확하다.

In [ ]:
import numpy as np


def quantize_asymmetric(w, bits=8):
    """비대칭 양자화 — 영점을 둔다"""
    qmin, qmax = 0, 2 ** bits - 1
    w_min, w_max = w.min(), w.max()
    scale = (w_max - w_min) / (qmax - qmin)
    zero_point = int(np.round(qmin - w_min / scale))
    q = np.round(w / scale + zero_point).clip(qmin, qmax).astype(int)
    return q, scale, zero_point


def dequantize_asymmetric(q, scale, zero_point):
    return (q - zero_point) * scale


# 한쪽으로 치우친 가중치
w_skewed = np.array([0.1, 0.15, 0.2, 0.3, 0.45, 0.5, 0.55, 0.6])

print("=" * 78)
print("대칭 vs 비대칭 — 치우친 분포에서")
print("=" * 78)
print(f"가중치: {w_skewed}")
print(f"  범위: {w_skewed.min()} ~ {w_skewed.max()}  (모두 양수)")
print()

bits = 4
q_sym, s_sym = quantize_symmetric(w_skewed, bits)
dq_sym = dequantize(q_sym, s_sym)

q_asym, s_asym, zp = quantize_asymmetric(w_skewed, bits)
dq_asym = dequantize_asymmetric(q_asym, s_asym, zp)

print(f"[{bits}비트]")
print(f"{'원본':<12}{'대칭 정수':<14}{'대칭 복원':<14}{'비대칭 정수':<14}{'비대칭 복원'}")
print("-" * 78)
for i in range(len(w_skewed)):
    print(f"{w_skewed[i]:<12.2f}{q_sym[i]:<14}{dq_sym[i]:<14.4f}"
          f"{q_asym[i]:<14}{dq_asym[i]:.4f}")
print("-" * 78)
print(f"{'평균 오차':<12}{'':<14}{np.abs(w_skewed-dq_sym).mean():<14.6f}"
      f"{'':<14}{np.abs(w_skewed-dq_asym).mean():.6f}")
print("-" * 78)
print()
print("[왜 차이가 나나]")
print(f"  대칭  : -{2**(bits-1)} ~ {2**(bits-1)-1} 범위를 쓰는데 음수는 하나도 없다")
print(f"          → 절반이 낭비된다")
print(f"  비대칭: 0 ~ {2**bits-1} 를 실제 범위 {w_skewed.min()}~{w_skewed.max()} 에 맞춘다")
print(f"          → 영점 {zp} 을 저장해야 하지만 훨씬 정확하다")

In [ ]:
import numpy as np


def quantize_grouped(w, bits=4, group_size=8):
    """그룹 단위 양자화 — 작은 묶음마다 scale 을 둔다"""
    flat = w.flatten()
    n_groups = int(np.ceil(len(flat) / group_size))
    dq = np.zeros_like(flat)
    scales = []

    for g in range(n_groups):
        start, end = g * group_size, min((g + 1) * group_size, len(flat))
        group = flat[start:end]
        q, s = quantize_symmetric(group, bits)
        dq[start:end] = dequantize(q, s)
        scales.append(s)

    return dq.reshape(w.shape), scales


print("=" * 78)
print("그룹 단위 양자화 — 이상치가 있을 때")
print("=" * 78)

np.random.seed(42)
w_normal = np.random.randn(64) * 0.1
w_outlier = w_normal.copy()
w_outlier[10] = 2.0      # 이상치 하나

print(f"가중치 64개, 대부분 ±0.3 범위")
print(f"  이상치 하나: {w_outlier[10]}")
print()

for name, w_test in [("이상치 없음", w_normal), ("이상치 있음", w_outlier)]:
    q, s = quantize_symmetric(w_test, 4)
    dq_whole = dequantize(q, s)
    dq_group, scales = quantize_grouped(w_test, 4, group_size=8)

    err_whole = np.abs(w_test - dq_whole).mean()
    err_group = np.abs(w_test - dq_group).mean()

    print(f"[{name}]")
    print(f"  전체 하나의 scale : {s:.6f}   평균 오차 {err_whole:.6f}")
    print(f"  그룹 8개씩        : scale {len(scales)}개  평균 오차 {err_group:.6f}")
    if err_whole > 0:
        print(f"  개선              : {err_whole/err_group:.1f}배")
    print()

print("-" * 78)
print("[이상치 하나가 전체를 망친다]")
print("  scale 은 최댓값으로 정해진다.")
print("  2.0 이라는 값 하나 때문에 나머지가 모두 거친 해상도로 표현된다.")
print()
print("[그룹 단위의 대가]")
print(f"  그룹마다 scale 을 저장해야 한다.")
print(f"  group_size=64 면 64개 가중치당 scale 1개 — 오버헤드 약 {32/64/4*100:.1f}%")
print()
print("  실무에서는 32 ~ 128 사이를 쓴다.")
print("  32장 8절의 'double quantization' 은 이 scale 마저 양자화하는 것이다.")

---

## 4. 비트 수와 품질 ★ — 이론편 22.5절

**여기가 이 장의 핵심이다.**

비트를 줄이면 크기는 선형으로 줄지만, **품질은 어느 지점에서 급격히 무너진다.**
실제 모델로 측정해 보자.

In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "distilgpt2"

print("모델 불러오는 중... (28장에서 받았다면 즉시)")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"완료 — {MODEL_NAME}, {n_params/1e6:.0f}M 파라미터")
print()

# 평가용 텍스트
eval_text = (
    "The quick brown fox jumps over the lazy dog. "
    "Machine learning is a field of study that gives computers "
    "the ability to learn without being explicitly programmed. "
    "Neural networks consist of layers of interconnected nodes."
)
eval_ids = tokenizer(eval_text, return_tensors="pt")


def perplexity(m, ids):
    """퍼플렉서티 — 낮을수록 좋다 (이론편 20.5절)"""
    m.eval()
    with torch.no_grad():
        loss = m(**ids, labels=ids["input_ids"]).loss
    return torch.exp(loss).item()


baseline_ppl = perplexity(model, eval_ids)
print(f"원본(FP32) 퍼플렉서티: {baseline_ppl:.3f}")
print()
print("[퍼플렉서티란]")
print("  '다음 토큰을 고를 때 평균 몇 개 중에서 헤매는가'로 볼 수 있다.")
print("  낮을수록 모델이 확신을 갖고 예측한다는 뜻이다.")

In [ ]:
import torch
import copy
import numpy as np


def quantize_tensor_torch(w, bits=8, group_size=None):
    """텐서를 양자화했다가 복원한다 (시뮬레이션)

    실제로는 정수로 저장하지만, 여기서는 품질 영향만 보므로
    양자화 → 복원까지 해서 오차만 남긴다.
    """
    qmax = 2 ** (bits - 1) - 1

    if group_size is None:
        scale = w.abs().max() / qmax
        if scale == 0:
            return w.clone()
        q = torch.round(w / scale).clamp(-qmax - 1, qmax)
        return q * scale

    # 그룹 단위
    original_shape = w.shape
    flat = w.flatten()
    pad = (group_size - len(flat) % group_size) % group_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, dtype=flat.dtype)])

    groups = flat.view(-1, group_size)
    scales = groups.abs().max(dim=1, keepdim=True).values / qmax
    scales = torch.where(scales == 0, torch.ones_like(scales), scales)
    q = torch.round(groups / scales).clamp(-qmax - 1, qmax)
    dq = (q * scales).flatten()

    if pad:
        dq = dq[:-pad]
    return dq.view(original_shape)


def quantize_model(base_model, bits, group_size=None, skip_embedding=False):
    """모델 전체를 양자화한다"""
    m = copy.deepcopy(base_model)
    errors = []

    with torch.no_grad():
        for name, param in m.named_parameters():
            if param.dim() < 2:          # 편향·정규화는 건너뛴다
                continue
            if skip_embedding and ("wte" in name or "wpe" in name):
                continue

            original = param.data.clone()
            param.data = quantize_tensor_torch(param.data, bits, group_size)
            errors.append((original - param.data).abs().mean().item())

    return m, float(np.mean(errors)) if errors else 0.0


print("=" * 78)
print("비트 수에 따른 품질 변화")
print("=" * 78)
print("(각 비트마다 모델을 복사해 양자화하므로 1분 정도 걸립니다)")
print()

bit_results = [{"bits": 32, "error": 0.0, "ppl": baseline_ppl,
                "size_gb": n_params * 4 / 1024**3}]

for bits in [8, 6, 5, 4, 3, 2]:
    qm, err = quantize_model(model, bits)
    ppl = perplexity(qm, eval_ids)
    bit_results.append({
        "bits": bits, "error": err, "ppl": ppl,
        "size_gb": n_params * bits / 8 / 1024**3,
    })
    del qm

print(f"{'비트':<10}{'평균 오차':<16}{'퍼플렉서티':<16}{'저하율':<16}{'크기 비율'}")
print("-" * 78)
for r in bit_results:
    degradation = (r["ppl"] / baseline_ppl - 1) * 100
    deg_str = "기준" if r["bits"] == 32 else f"{degradation:+.1f}%"
    ratio = r["size_gb"] / bit_results[0]["size_gb"]
    print(f"{r['bits']:<10}{r['error']:<16.6f}{r['ppl']:<16.2f}"
          f"{deg_str:<16}{ratio:.3f}")
print("-" * 78)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

bits_list = [r["bits"] for r in bit_results]
ppls = [r["ppl"] for r in bit_results]
errors = [r["error"] for r in bit_results]
sizes = [r["size_gb"] / bit_results[0]["size_gb"] for r in bit_results]

# --- 왼쪽: 퍼플렉서티 (로그) ---
ax = axes[0]
ax.plot(bits_list, ppls, marker="o", markersize=8, linewidth=2.5,
        color="#DC2626")
ax.axhline(baseline_ppl, color="#0D9488", linestyle="--", linewidth=1.5)
ax.text(20, baseline_ppl * 1.3, "원본 수준", fontsize=8, color="#0D9488")

# 절벽 지점 표시
for i, r in enumerate(bit_results):
    if r["bits"] < 32 and r["ppl"] > baseline_ppl * 2:
        ax.axvline(r["bits"], color="#EA580C", linestyle=":", linewidth=2)
        ax.text(r["bits"] + 0.4, max(ppls) * 0.3,
                f"{r['bits']}비트부터\n급격히 나빠짐", fontsize=8, color="#EA580C")
        break

ax.set_yscale("log")
ax.set_xlabel("비트 수")
ax.set_ylabel("퍼플렉서티 (로그, 낮을수록 좋음)")
ax.set_title("품질에는 절벽이 있다")
ax.invert_xaxis()
ax.grid(alpha=0.3, which="both")

# --- 오른쪽: 크기 vs 품질 ---
ax = axes[1]
for r in bit_results:
    color = "#0D9488" if r["ppl"] < baseline_ppl * 1.5 else "#DC2626"
    ax.scatter(r["size_gb"] / bit_results[0]["size_gb"] * 100,
               r["ppl"], s=140, color=color,
               edgecolors="white", linewidth=1.5, zorder=3)
    ax.annotate(f"{r['bits']}비트",
                (r["size_gb"] / bit_results[0]["size_gb"] * 100, r["ppl"]),
                textcoords="offset points", xytext=(8, 6), fontsize=8)

ax.set_yscale("log")
ax.set_xlabel("모델 크기 (원본 대비 %)")
ax.set_ylabel("퍼플렉서티 (로그)")
ax.set_title("크기와 품질의 맞바꿈")
ax.grid(alpha=0.3, which="both")

plt.tight_layout()
plt.show()

# 절벽 찾기
cliff = None
for r in bit_results[1:]:
    if r["ppl"] > baseline_ppl * 2:
        cliff = r["bits"]
        break

print("=" * 78)
print("측정 결과 해석")
print("=" * 78)
print()
if cliff:
    safe = [r["bits"] for r in bit_results[1:] if r["ppl"] < baseline_ppl * 1.5]
    print(f"  품질을 유지하는 비트: {safe}")
    print(f"  급격히 나빠지는 비트: {cliff} 이하")
print()
print("[왜 절벽이 생기나]")
print("  가중치 분포는 대체로 종 모양이다 (평균 0 근처에 몰림).")
print("  비트가 줄면 그 좁은 구간을 표현할 단계가 부족해진다.")
print()
print("  8비트: 256단계 — 충분")
print("  4비트: 16단계 — 대부분의 가중치가 같은 값으로 뭉개짐")
print()
print("[그런데 실제 4비트 모델은 쓸 만하다]")
print("  단순 양자화가 아니라 그룹 단위·NF4 같은 기법을 쓰기 때문이다.")
print("  다음 셀에서 확인한다.")

In [ ]:
import torch
import numpy as np

print("=" * 78)
print("그룹 단위 양자화로 4비트 구하기")
print("=" * 78)
print("(3절에서 다룬 기법을 실제 모델에 적용한다)")
print()

group_results = []
for group_size in [None, 128, 64, 32]:
    qm, err = quantize_model(model, 4, group_size=group_size)
    ppl = perplexity(qm, eval_ids)
    group_results.append({
        "group": "전체" if group_size is None else str(group_size),
        "error": err, "ppl": ppl,
    })
    del qm

print(f"{'그룹 크기':<16}{'평균 오차':<16}{'퍼플렉서티':<16}{'원본 대비'}")
print("-" * 78)
print(f"{'FP32 (원본)':<16}{0:<16.6f}{baseline_ppl:<16.2f}기준")
for r in group_results:
    deg = (r["ppl"] / baseline_ppl - 1) * 100
    print(f"{r['group']:<16}{r['error']:<16.6f}{r['ppl']:<16.2f}{deg:+.1f}%")
print("-" * 78)
print()

best_group = min(group_results, key=lambda r: r["ppl"])
worst = max(group_results, key=lambda r: r["ppl"])
print(f"최선: 그룹 {best_group['group']} (퍼플렉서티 {best_group['ppl']:.2f})")
print(f"최악: 그룹 {worst['group']} (퍼플렉서티 {worst['ppl']:.2f})")
if worst["ppl"] > 0:
    print(f"개선: {worst['ppl']/best_group['ppl']:.1f}배")
print()
print("[같은 4비트인데 그룹 크기만으로 큰 차이가 난다]")
print("  이것이 실제 4비트 모델이 쓸 만한 이유다.")
print("  32장 8절의 QLoRA 도 그룹 단위 NF4 를 쓴다.")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4.2))
labels = [r["group"] for r in group_results]
ppls_g = [r["ppl"] for r in group_results]
colors_g = ["#DC2626" if p > baseline_ppl * 2 else "#0D9488" for p in ppls_g]
bars = ax.bar(range(len(labels)), ppls_g, color=colors_g)
ax.axhline(baseline_ppl, color="#1E40AF", linestyle="--", linewidth=2)
ax.text(len(labels) - 0.6, baseline_ppl * 1.2, "원본", fontsize=9, color="#1E40AF")
for b, p in zip(bars, ppls_g):
    ax.text(b.get_x() + b.get_width()/2, p * 1.05, f"{p:.0f}",
            ha="center", fontsize=9)
ax.set_yscale("log")
ax.set_xticks(range(len(labels)))
ax.set_xticklabels([f"그룹 {l}" for l in labels])
ax.set_ylabel("퍼플렉서티 (로그)")
ax.set_title("4비트 양자화 — 그룹 크기의 영향")
ax.grid(axis="y", alpha=0.3, which="both")
plt.tight_layout()
plt.show()

---

## 5. 실제 모델 양자화 — 이론편 22.5절

지금까지는 **시뮬레이션**이었다 (양자화했다가 다시 실수로 복원).
실제로는 **정수로 저장하고 정수로 계산**한다.

PyTorch의 동적 양자화를 써 보자.

In [ ]:
import torch
import torch.nn as nn
import io
import time
import warnings

print("=" * 78)
print("PyTorch 동적 양자화")
print("=" * 78)
print()


def model_file_size_mb(m):
    """저장했을 때의 크기"""
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.tell() / 1024**2


print("[동적 양자화란]")
print("  가중치는 미리 INT8 로 바꾸고,")
print("  활성값은 실행 중에 그때그때 양자화한다.")
print("  → 보정(calibration) 데이터가 필요 없어 간단하다")
print()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    t0 = time.time()
    quantized = torch.quantization.quantize_dynamic(
        model, {nn.Linear}, dtype=torch.qint8)
    convert_time = time.time() - t0

print(f"변환 시간: {convert_time:.1f}초")
print()

size_fp32 = model_file_size_mb(model)
size_int8 = model_file_size_mb(quantized)

print(f"{'모델':<20}{'저장 크기':<20}{'퍼플렉서티'}")
print("-" * 78)
print(f"{'FP32 원본':<20}{size_fp32:<20.1f}{perplexity(model, eval_ids):.2f}")
print(f"{'INT8 동적':<20}{size_int8:<20.1f}{perplexity(quantized, eval_ids):.2f}")
print("-" * 78)
print()

# 어떤 층이 실제로 양자화되었는지 확인
n_linear = sum(1 for m in model.modules() if isinstance(m, nn.Linear))
n_conv1d = sum(1 for m in model.modules() if type(m).__name__ == "Conv1D")

print("[예상과 다를 수 있다]")
print(f"  nn.Linear 층: {n_linear}개")
print(f"  Conv1D 층   : {n_conv1d}개")
print()
if n_conv1d > n_linear:
    print("  GPT-2 계열은 nn.Linear 대신 Conv1D 를 쓴다.")
    print("  → quantize_dynamic({nn.Linear}) 이 거의 적용되지 않는다")
    print()
    print("  이런 이유로 실무에서는 모델 구조에 맞는 전용 도구를 쓴다 (7절).")

In [ ]:
import torch
import time

print("=" * 78)
print("추론 속도 비교")
print("=" * 78)

prompt = "The future of artificial intelligence"
inputs = tokenizer(prompt, return_tensors="pt")

print(f"{'모델':<20}{'20토큰 생성':<20}{'토큰/초'}")
print("-" * 78)

speed_results = {}
for name, m in [("FP32", model), ("INT8 동적", quantized)]:
    times = []
    for _ in range(3):
        t0 = time.time()
        with torch.no_grad():
            m.generate(**inputs, max_new_tokens=20, do_sample=False,
                       pad_token_id=tokenizer.eos_token_id)
        times.append(time.time() - t0)
    avg = sum(times) / len(times)
    speed_results[name] = avg
    print(f"{name:<20}{avg:<20.2f}{20/avg:.1f}")

print("-" * 78)
if speed_results["INT8 동적"] > 0:
    print(f"속도 차이: {speed_results['FP32']/speed_results['INT8 동적']:.2f}배")
print()
print("[CPU 에서 양자화가 속도를 높이는 이유]")
print("  정수 연산이 부동소수점보다 빠르다.")
print("  메모리 대역폭도 덜 쓴다 (35장 4절의 메모리 병목).")
print()
print("[GPU 에서는 다를 수 있다]")
print("  GPU 는 FP16 연산이 매우 빠르므로, 양자화가 항상 빠른 것은 아니다.")
print("  GPU 에서 양자화의 주된 목적은 **속도가 아니라 메모리**다.")

In [ ]:
import torch

print("=" * 78)
print("생성 결과 비교 — 품질을 눈으로")
print("=" * 78)

prompts = [
    "The capital of France is",
    "Machine learning is",
]

for prompt in prompts:
    inputs = tokenizer(prompt, return_tensors="pt")
    print(f"\n프롬프트: {prompt}")
    print("-" * 78)
    for name, m in [("FP32", model), ("INT8", quantized)]:
        with torch.no_grad():
            out = m.generate(**inputs, max_new_tokens=15, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0]).replace("\n", " ")
        print(f"  {name:<8}{text[:66]}")

print()
print("=" * 78)
print("[품질 평가의 어려움]")
print("  퍼플렉서티는 참고 지표일 뿐이다.")
print("  실제로 쓸 만한지는 **사람이 보거나 과제로 평가**해야 한다.")
print()
print("  34장에서 다룰 평가 자동화가 여기서 필요해진다.")

---

## 6. 어떤 층을 양자화할까 — 이론편 22.5절

**모든 층을 똑같이 다루면 안 된다.**

| 층 | 양자화 | 이유 |
|---|---|---|
| 임베딩 | 신중히 | 모든 토큰의 표현 기반 |
| Attention (Q,K,V,O) | 가능 | 파라미터가 많음 |
| FFN | 가능 | 파라미터가 가장 많음 (27장 5절) |
| LayerNorm | **하지 않음** | 파라미터가 적고 민감 |
| 마지막 출력층 | 신중히 | 어휘 전체에 영향 |

In [ ]:
import torch
import numpy as np

print("=" * 78)
print("층별 양자화 민감도")
print("=" * 78)
print("(층 그룹별로 하나씩만 양자화해 영향을 본다)")
print()


def quantize_selective(base_model, bits, target_keywords):
    """지정한 층만 양자화한다"""
    m = copy.deepcopy(base_model)
    n_quantized = 0
    with torch.no_grad():
        for name, param in m.named_parameters():
            if param.dim() < 2:
                continue
            if any(kw in name for kw in target_keywords):
                param.data = quantize_tensor_torch(param.data, bits)
                n_quantized += 1
    return m, n_quantized


layer_groups = {
    "임베딩 (wte, wpe)": ["wte", "wpe"],
    "Attention (attn)": ["attn"],
    "FFN (mlp)": ["mlp"],
    "전부": ["wte", "wpe", "attn", "mlp"],
}

BITS = 4
print(f"{BITS}비트로 양자화 (그룹 없음)")
print()
print(f"{'양자화 대상':<26}{'층 수':<12}{'퍼플렉서티':<16}{'저하율'}")
print("-" * 78)
print(f"{'없음 (원본)':<26}{0:<12}{baseline_ppl:<16.2f}기준")

sensitivity = []
for label, keywords in layer_groups.items():
    qm, n_q = quantize_selective(model, BITS, keywords)
    ppl = perplexity(qm, eval_ids)
    deg = (ppl / baseline_ppl - 1) * 100
    sensitivity.append({"label": label, "ppl": ppl, "deg": deg, "n": n_q})
    print(f"{label:<26}{n_q:<12}{ppl:<16.2f}{deg:+.1f}%")
    del qm

print("-" * 78)
print()

most_sensitive = max(sensitivity[:-1], key=lambda s: s["deg"])
print(f"가장 민감한 층: {most_sensitive['label']} ({most_sensitive['deg']:+.1f}%)")
print()
print("[실무의 접근]")
print("  민감한 층은 높은 정밀도(8비트 또는 FP16)로 두고")
print("  나머지만 4비트로 낮추는 **혼합 정밀도**를 쓴다.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 4.2))

labels_s = [s["label"] for s in sensitivity]
degs = [s["deg"] for s in sensitivity]
colors_s = ["#DC2626" if d > 500 else ("#EA580C" if d > 100 else "#0D9488")
            for d in degs]

bars = ax.barh(range(len(labels_s)), degs, color=colors_s)
for b, d in zip(bars, degs):
    ax.text(d * 1.02, b.get_y() + b.get_height()/2, f"{d:+.0f}%",
            va="center", fontsize=9)
ax.set_yticks(range(len(labels_s)))
ax.set_yticklabels(labels_s, fontsize=9)
ax.set_xlabel("퍼플렉서티 저하율 (%)")
ax.set_title(f"{BITS}비트 양자화 — 층별 민감도")
ax.set_xscale("symlog")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print("=" * 78)
print("혼합 정밀도 예시")
print("=" * 78)
print()

# 민감한 층은 8비트, 나머지는 4비트
mixed = copy.deepcopy(model)
with torch.no_grad():
    for name, param in mixed.named_parameters():
        if param.dim() < 2:
            continue
        if "wte" in name or "wpe" in name:
            param.data = quantize_tensor_torch(param.data, 8, group_size=64)
        else:
            param.data = quantize_tensor_torch(param.data, 4, group_size=64)

mixed_ppl = perplexity(mixed, eval_ids)

print(f"{'구성':<34}{'퍼플렉서티':<16}{'저하율'}")
print("-" * 78)
print(f"{'FP32 원본':<34}{baseline_ppl:<16.2f}기준")
print(f"{'전부 4비트 (그룹 64)':<34}{group_results[2]['ppl']:<16.2f}"
      f"{(group_results[2]['ppl']/baseline_ppl-1)*100:+.1f}%")
print(f"{'임베딩 8비트 + 나머지 4비트':<34}{mixed_ppl:<16.2f}"
      f"{(mixed_ppl/baseline_ppl-1)*100:+.1f}%")
print("-" * 78)
print()
print("임베딩만 8비트로 올려도 품질이 개선된다.")
print("  임베딩은 파라미터 비중이 크지만, 정밀도가 특히 중요한 부분이다.")
print()
del mixed

---

## 7. 실무 도구 — 이론편 22.5절

**직접 구현하지 않는다.** 최적화된 도구들이 있다.

In [ ]:
print("=" * 78)
print("양자화 도구 비교")
print("=" * 78)
print()
print(f"{'도구':<20}{'용도':<28}{'특징'}")
print("-" * 78)
tools = [
    ("bitsandbytes", "GPU 학습·추론 (QLoRA)", "4비트 NF4, 32장에서 사용"),
    ("GGUF (llama.cpp)", "CPU 추론", "다양한 비트, 노트북에서도"),
    ("AWQ", "GPU 추론", "활성값 인식 양자화"),
    ("GPTQ", "GPU 추론", "층별 최적화, 보정 필요"),
    ("torch.quantization", "CPU 추론", "PyTorch 내장 (5절)"),
]
for a, b, c in tools:
    print(f"{a:<20}{b:<28}{c}")
print("-" * 78)
print()

print("[bitsandbytes — 32장에서 다룬 것]")
print()
code_bnb = [
    "from transformers import AutoModelForCausalLM, BitsAndBytesConfig",
    "import torch",
    "",
    "bnb_config = BitsAndBytesConfig(",
    "    load_in_4bit=True,",
    "    bnb_4bit_quant_type='nf4',              # 정규분포에 맞춘 4비트",
    "    bnb_4bit_compute_dtype=torch.bfloat16,  # 계산은 16비트",
    "    bnb_4bit_use_double_quant=True,         # scale 도 양자화",
    ")",
    "",
    "model = AutoModelForCausalLM.from_pretrained(",
    "    MODEL_NAME, quantization_config=bnb_config, device_map='auto')",
]
for line in code_bnb:
    print("  " + line)

print()
print("  주의: CUDA 가 필요하다. CPU 에서는 동작하지 않는다.")
print()
print("[GGUF — CPU 에서 쓸 때]")
print()
code_gguf = [
    "# 1) 모델을 GGUF 형식으로 변환 (llama.cpp 도구 사용)",
    "python convert-hf-to-gguf.py <모델경로> --outfile model.gguf",
    "",
    "# 2) 양자화",
    "./llama-quantize model.gguf model-q4.gguf Q4_K_M",
    "",
    "# 3) 실행",
    "./llama-cli -m model-q4.gguf -p '프롬프트'",
]
for line in code_gguf:
    print("  " + line)

In [ ]:
print("=" * 78)
print("GGUF 양자화 방식 이름 읽는 법")
print("=" * 78)
print()
print("  Q4_K_M 같은 이름이 무엇을 뜻하는지")
print()
print(f"{'부분':<12}{'뜻'}")
print("-" * 78)
print(f"{'Q4':<12}4비트 양자화")
print(f"{'K':<12}K-quant 방식 (그룹 단위 + 최적화)")
print(f"{'M':<12}크기 등급 (S=Small, M=Medium, L=Large)")
print("-" * 78)
print()
print("자주 쓰이는 조합")
print(f"{'이름':<14}{'대략 비트':<14}{'특징'}")
print("-" * 78)
print(f"{'Q8_0':<14}{'8.0':<14}거의 원본 품질")
print(f"{'Q6_K':<14}{'6.6':<14}품질 좋음")
print(f"{'Q5_K_M':<14}{'5.7':<14}균형")
print(f"{'Q4_K_M':<14}{'4.8':<14}**가장 많이 쓰임**")
print(f"{'Q4_K_S':<14}{'4.6':<14}조금 더 작음")
print(f"{'Q3_K_M':<14}{'3.9':<14}품질 저하 눈에 띔")
print(f"{'Q2_K':<14}{'3.0':<14}권장하지 않음")
print("-" * 78)
print()
print("[왜 Q4_K_M 이 표준이 되었나]")
print("  4절에서 측정했듯 4비트가 크기 대비 품질의 균형점이다.")
print("  K-quant 가 3절의 그룹 단위 기법을 정교하게 적용해")
print("  단순 4비트보다 훨씬 나은 품질을 낸다.")
print()
print("[8GB GPU 기준 — 32장 9절 참조]")
print(f"{'모델':<14}{'Q4_K_M 크기':<20}{'가능 여부'}")
print("-" * 78)
for n, name in [(3, "3B"), (7, "7B"), (13, "13B"), (30, "30B")]:
    size = n * 4.8 / 8
    print(f"{name:<14}{size:>7.1f} GB          {'가능' if size < 6.5 else '어려움'}")
print("-" * 78)

---

## 8. 무엇을 얻고 무엇을 잃나 — 이론편 22.5절

In [ ]:
import numpy as np

print("=" * 78)
print("양자화의 득실")
print("=" * 78)
print()
print(f"{'항목':<24}{'효과':<30}{'주의'}")
print("-" * 78)
tradeoffs = [
    ("메모리", "비트 수에 비례해 감소", "가장 확실한 이득"),
    ("모델 크기", "저장·전송 용량 감소", "배포에 유리"),
    ("CPU 속도", "정수 연산이 빠름", "구현에 따라 다름"),
    ("GPU 속도", "항상 빠르지는 않음", "역양자화 비용"),
    ("품질", "비트에 따라 저하", "**절벽 주의** (4절)"),
    ("학습", "직접 학습 불가", "QLoRA 로 우회 (32장)"),
]
for a, b, c in tradeoffs:
    print(f"{a:<24}{b:<30}{c}")
print("-" * 78)
print()

print("[가장 중요한 것]")
print()
print("  '7B 를 4비트로' vs '1.5B 를 FP16 으로'")
print()
print("  대체로 **전자가 낫다.** 모델 크기의 이득이 양자화 손실보다 크다.")
print("  다만 **측정해서 확인**해야 한다 (34장에서 다룰 평가).")
print()
print("=" * 78)
print("선택 가이드")
print("=" * 78)
print(f"{'상황':<30}{'권장':<20}{'이유'}")
print("-" * 78)
choices = [
    ("GPU 메모리가 충분",        "FP16",              "품질 최우선"),
    ("메모리가 약간 부족",       "INT8",              "품질 거의 유지"),
    ("메모리가 많이 부족",       "INT4 (그룹 단위)",   "실용적 균형"),
    ("CPU 에서 실행",           "GGUF Q4_K_M",       "CPU 최적화"),
    ("파인튜닝이 필요",         "QLoRA",             "32장 8절"),
    ("품질이 절대적",           "양자화하지 않음",     "—"),
]
for a, b, c in choices:
    print(f"{a:<30}{b:<20}{c}")
print("-" * 78)

In [ ]:
print("=" * 78)
print("양자화 전에 확인할 것")
print("=" * 78)
print()
checklist = [
    "정말 메모리가 부족한가 (배치 축소로 해결되지 않나)",
    "더 작은 모델로 충분하지 않은가",
    "품질 평가 기준이 준비되어 있는가",
    "양자화 후 실제 과제로 측정할 계획이 있는가",
    "GPU 인가 CPU 인가 (도구가 다르다)",
    "추론만 할 것인가, 학습도 할 것인가 (QLoRA 필요)",
]
for i, item in enumerate(checklist, 1):
    print(f"  {i}. □ {item}")

print()
print("-" * 78)
print("[가장 흔한 실수]")
print()
print("  퍼플렉서티만 보고 '괜찮다'고 판단하는 것")
print()
print("  퍼플렉서티는 다음 토큰 예측 능력만 본다.")
print("  실제 과제(요약·번역·추론)에서는 더 크게 나빠질 수 있다.")
print()
print("  → 34장에서 다룰 **과제 기반 평가**가 필요하다")
print()
print("[이 장에서 확인한 것]")
print(f"  8비트  : 품질 거의 유지")
print(f"  4비트  : 그룹 단위를 쓰면 실용적")
print(f"  3비트 이하: 급격히 무너짐")
print()
print("  다만 이것은 작은 모델(82M)의 결과다.")
print("  큰 모델일수록 양자화에 강한 경향이 있다 — 여유가 더 많기 때문이다.")

---

## 9. 정리

### 확인한 이론편 값

| 이론편 절 | 내용 | 결과 |
|---|---|---|
| **22.5** | **8비트 [-127, -48, 0, 40, 111]** | **일치** ✓ |
| **22.5** | **4비트 [-7, -3, 0, 2, 6]** | **일치** ✓ |
| 22.5 | 비트 수와 모델 크기 | 계산 ✓ |
| 22.5 | 품질 저하의 절벽 | 측정 ✓ |
| 22.5 | 그룹 단위의 효과 | 측정 ✓ |

### 양자화 방식

```python
# 대칭 양자화
scale = |w|.max() / (2^(bits-1) - 1)
q = round(w / scale)
w_hat = q * scale

# 그룹 단위 — 작은 묶음마다 scale
for group in chunks(w, group_size):
    quantize(group)
```

### 기억할 것

| 항목 | 요점 |
|---|---|
| 대칭 vs 비대칭 | 치우친 분포는 비대칭이 정확 |
| **그룹 단위** | **이상치의 영향을 막는다** |
| 품질 절벽 | 8비트는 안전, 3비트 이하는 위험 |
| 층별 민감도 | 임베딩·LayerNorm은 신중히 |
| 혼합 정밀도 | 민감한 층만 높은 비트로 |
| GPU vs CPU | GPU는 메모리, CPU는 속도가 주 이득 |
| Q4_K_M | GGUF의 사실상 표준 |
| 평가 | 퍼플렉서티만으로 판단 금지 |

### 32장과 이어지는 지점

32장 8절에서 QLoRA를 다루며 이렇게 설명했다.

> **원본 가중치는 4비트로 얼리고, LoRA 어댑터만 16비트로 학습한다.**

이 장에서 **왜 그렇게 나누는지** 확인했다.
얼린 가중치는 읽기만 하므로 정밀도를 낮춰도 되지만,
학습되는 부분은 미세한 갱신이 쌓여야 하므로 정밀해야 한다.

### 다음 장

**34. DPO — 선호로 학습하기** — 이 장에서 "퍼플렉서티만으로는 부족하다"고 했다.
그렇다면 무엇으로 평가할 것인가. LLM-as-Judge를 포함한 방법들을 다룬다.